# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`  
This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all available record sets (tables) in the FAIR² dataset, along with their `@id` values, and then list example fields and columns for one record set.

In [ ]:
# List record sets in the dataset
print("Available record sets:")
for rs in dataset.record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# Explore fields and columns for each record set
for rs in dataset.record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name}, @id: {field.id}")
        if getattr(field, 'columns', None):
            for col in field.columns:
                print(f"        * Column: {col.name}, @id: {col.id}")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references use the `@id` fields as required.

We will extract data from each available record set and print its columns and a preview.

In [ ]:
# Prepare to extract data from all record sets by their `@id`
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecord Set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# Choose a record set to focus on—here, we select the first if available
if record_sets_ids:
    selected_record_set_id = record_sets_ids[0]
    print(f"\nSelected record set for EDA: {selected_record_set_id}")
    df_selected = dataframes[selected_record_set_id]
    print(df_selected.head())
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
This section applies common data processing steps, such as filtering numeric records and normalizing numeric fields. All field and column references are by their `@id`.

**If you see KeyErrors or empty outputs, check the column `@id` values printed above and update variable assignments below accordingly.**

In [ ]:
# --- Begin EDA ---
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Available columns in record set {selected_record_set_id}:")
    print(df.columns.tolist())

    # Select a numeric field (update this to a valid '@id' found above if needed)
    # For illustration, pick the first column which appears numeric
    numeric_column_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_column_id = col
            break

    if numeric_column_id is not None:
        print(f"\nUsing numeric field '@id': {numeric_column_id}")
        threshold = df[numeric_column_id].mean() if pd.notna(df[numeric_column_id].mean()) else 0

        # Filter records with values above threshold
        filtered_df = df[df[numeric_column_id] > threshold]
        print(f"\nFiltered records where {numeric_column_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field (z-score normalization)
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_column_id}_normalized"] = (
            (filtered_df[numeric_column_id] - filtered_df[numeric_column_id].mean()) /
            filtered_df[numeric_column_id].std()
        )
        print(f"\nNormalized {numeric_column_id} for filtered records:")
        print(filtered_df[[numeric_column_id, f"{numeric_column_id}_normalized"]].head())

        # Attempt to group by a categorical field
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and len(df[col].unique()) < 20 and col != numeric_column_id:
                group_field_id = col
                break

        if group_field_id:
            print(f"\nGrouping by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_column_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("\nNo suitable categorical field found for grouping.")
    else:
        print("\nNo numeric field found for EDA in selected record set.")
else:
    print("No record set was selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following cell dynamically displays a histogram for the selected numeric field (if available) in the chosen record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and 'numeric_column_id' in locals() and numeric_column_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_column_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_column_id} in {selected_record_set_id}")
    plt.xlabel(numeric_column_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using the Croissant schema and explored its metadata, record sets, and fields. We proceeded with basic data extraction into pandas DataFrames by referencing all entities via their `@id`. We also demonstrated simple filtering, normalization, grouping, and visualization for numeric fields. This provides a reproducible framework for deeper analysis of socio-demographic and adoption predictors in rangeland management practices in Northern Kenya.
